# Gullak: knowledge tracing — synthetic checks and a real-data benchmark

Increment 1: validate quiz answer rows and generate 96 synthetic answers (12 learners, four submissions, two questions).
Increment 2: simulate synthetic learners from known BKT parameters, fit per-concept BKT by EM on a learner-separated training split, check held-out next-answer prediction against a constant baseline, and export a draft parameter file labelled synthetic. Increment 3: benchmark BKT, BKT with forgetting, PFA and DKT (an LSTM) on **real** student answers from ASSISTments 2009–10, with 5-fold learner-separated cross-validation.
No app integration; quiz graph pass marks remain the only unlock rule.

**To run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. No installs, keys or Drive mount. Increments 1–2 take seconds; increment 3 downloads the 64 MB dataset and takes about 10 minutes (most of it DKT on the GPU). Two files download at the end: `knowledge-tracing-params.synthetic-draft.json` and `knowledge-tracing-benchmark.assistments-2009.json`.

The nine input columns match the latest R3 contract: event_id, learner_id, topic_id, concept_id, question_id, correct, answered_at, attempt_no, content_version. No question_index column. Content version is SHA-256 of exact topic-content UTF-8 bytes. Use an immutable content archive supplied by R3 to obtain question order and concepts; never reconstruct history from the current answer key.

Each submission shares attempt_no (the topic's quiz_attempts after submission). Within it, questions are ordered by position in pinned content. Missing historical content or ambiguous positions fail closed. The synthetic content below is an engineering fixture with the real money-in / money-out concept IDs, not a reviewed lesson.

Synthetic labels live in notebook and dataset metadata, not extra input columns. Keep future real learner rows out of this publicly shared notebook. Story safe/risky/harmful verdicts are excluded from BKT for now.

In [ ]:
import hashlib
import json
import random
import re
from datetime import datetime, timedelta, timezone
from uuid import UUID

LABEL = "SYNTHETIC — NOT REAL STUDENT DATA"
FIELDS = {"event_id", "learner_id", "topic_id", "concept_id", "question_id",
          "correct", "answered_at", "attempt_no", "content_version"}
# Synthetic teaching fixture, NOT an approved/published money-basics lesson.
SYNTHETIC_CONTENT = {
    "topicId": "money-basics", "locale": "en",
    "title": "SYNTHETIC engineering fixture", "summary": LABEL,
    "variants": [{"id": "synthetic-check", "format": "quiz", "questions": [
        {"id": "money-in", "conceptId": "money-in", "prompt": "Gift arrives: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "in", "explanation": "A gift adds money."},
        {"id": "money-out", "conceptId": "money-out", "prompt": "Paying a bill: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "out", "explanation": "A payment takes money away."}
    ]}]
}
# Hash these exact UTF-8 bytes. Never reserialize real content to calculate its hash.
SYNTHETIC_CONTENT_RAW = json.dumps(SYNTHETIC_CONTENT, ensure_ascii=False, separators=(",", ":"))


def synthetic_content_hash():
    return hashlib.sha256(SYNTHETIC_CONTENT_RAW.encode("utf-8")).hexdigest()


def validate_answer_rows(rows, registry):
    """Validate nine-field rows against a trusted, version-pinned question registry.
    Registry keys: (topic_id, content_version, question_id).
    Values: {"concept_id": canonical slug, "position": content question order}.
    Return sorted copies; no guessing, grading, mastery update or mutation.
    """
    if not isinstance(rows, list) or not rows:
        raise ValueError("expected a non-empty list of quiz answers")
    parsed, event_ids, positions, questions, versions = [], set(), set(), set(), {}
    for index, raw in enumerate(rows):
        if not isinstance(raw, dict) or set(raw) != FIELDS:
            raise ValueError(f"row {index}: expected exactly the nine quiz_answers fields")
        row = dict(raw)
        for field in ("event_id", "learner_id"):
            value = row[field]
            if not isinstance(value, str):
                raise ValueError(f"row {index}: invalid {field}")
            try:
                normalized = str(UUID(value))
            except ValueError as error:
                raise ValueError(f"row {index}: invalid {field}") from error
            row[field] = normalized
        for field in ("topic_id", "concept_id", "question_id"):
            if not isinstance(row[field], str) or not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", row[field]):
                raise ValueError(f"row {index}: invalid {field} slug")
        if type(row["correct"]) is not bool:
            raise ValueError(f"row {index}: correct must be boolean")
        if type(row["attempt_no"]) is not int or not 1 <= row["attempt_no"] <= 2147483647:
            raise ValueError(f"row {index}: attempt_no must be a positive Postgres integer")
        if not isinstance(row["content_version"], str) or not re.fullmatch(r"[a-f0-9]{64}", row["content_version"]):
            raise ValueError(f"row {index}: content_version must be lowercase SHA-256 hex")
        if not isinstance(row["answered_at"], str):
            raise ValueError(f"row {index}: timestamp must be text")
        try:
            stamp = datetime.fromisoformat(row["answered_at"].replace("Z", "+00:00"))
        except ValueError as error:
            raise ValueError(f"row {index}: invalid timestamp") from error
        if stamp.tzinfo is None or stamp.utcoffset() is None:
            raise ValueError(f"row {index}: timestamp must include a timezone")
        row["answered_at"] = stamp.astimezone(timezone.utc).isoformat()
        definition = registry.get((row["topic_id"], row["content_version"], row["question_id"]))
        if not isinstance(definition, dict) or definition.get("concept_id") != row["concept_id"]:
            raise ValueError(f"row {index}: unknown pinned question or concept mismatch")
        position = definition.get("position")
        if type(position) is not int or position < 0:
            raise ValueError(f"row {index}: missing trusted question position")
        submission = (row["learner_id"], row["topic_id"], row["attempt_no"])
        if submission in versions and versions[submission] != row["content_version"]:
            raise ValueError(f"row {index}: mixed content versions within a submission")
        versions[submission] = row["content_version"]
        position_key = (*submission, position)
        question_key = (*submission, row["question_id"])
        if row["event_id"] in event_ids or position_key in positions or question_key in questions:
            raise ValueError(f"row {index}: duplicate event, question or ambiguous content order")
        event_ids.add(row["event_id"])
        positions.add(position_key)
        questions.add(question_key)
        parsed.append((row, position))
    parsed.sort(key=lambda pair: (pair[0]["learner_id"], pair[0]["topic_id"], pair[0]["attempt_no"], pair[1]))
    # A whole submission may share a server timestamp. Check between submissions,
    # not within one: content order, rather than timestamp, orders its answers.
    ranges = {}
    for row, _ in parsed:
        key = (row["learner_id"], row["topic_id"], row["attempt_no"])
        stamp = datetime.fromisoformat(row["answered_at"])
        old = ranges.get(key, (stamp, stamp))
        ranges[key] = (min(old[0], stamp), max(old[1], stamp))
    previous = {}
    for (learner, topic, _), (earliest, latest) in sorted(ranges.items()):
        key = (learner, topic)
        if key in previous and earliest < previous[key]:
            raise ValueError("timestamps contradict quiz attempt order")
        previous[key] = latest
    return [row for row, _ in parsed]


def generate_synthetic_responses(seed=42):
    """Fixed small engineering fixture; no model training or accuracy claims."""
    if type(seed) is not int:
        raise ValueError("seed must be an integer")
    rng = random.Random(seed)
    version = synthetic_content_hash()
    quiz = SYNTHETIC_CONTENT["variants"][0]["questions"]
    registry = {("money-basics", version, q["id"]):
                {"concept_id": q["conceptId"], "position": position}
                for position, q in enumerate(quiz)}
    rows = []
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    for learner_index in range(12):
        learner = str(UUID(int=rng.getrandbits(128), version=4))
        for attempt in range(1, 5):
            timestamp = (start + timedelta(days=attempt - 1, minutes=learner_index)).isoformat()
            for question in quiz:
                rows.append({
                    "event_id": str(UUID(int=rng.getrandbits(128), version=4)),
                    "learner_id": learner, "topic_id": "money-basics",
                    "concept_id": question["conceptId"], "question_id": question["id"],
                    "correct": rng.random() < 0.25 + attempt * 0.12,
                    "answered_at": timestamp, "attempt_no": attempt, "content_version": version,
                })
    metadata = {"label": LABEL, "synthetic": True, "production_approved": False,
                "seed": seed, "content_version": version,
                "generator": "fixed engineering fixture; not a calibrated learner model"}
    return rows, registry, metadata


In [ ]:
import copy
import unittest

class SyntheticInputTests(unittest.TestCase):
    def setUp(self):
        self.rows, self.registry, self.metadata = generate_synthetic_responses(seed=42)

    def test_valid_rows_match_exact_contract(self):
        result = validate_answer_rows(self.rows, self.registry)
        self.assertEqual(len(result), 96)
        self.assertEqual(set(result[0]), {
            "event_id", "learner_id", "topic_id", "concept_id", "question_id",
            "correct", "answered_at", "attempt_no", "content_version",
        })

    def test_seed_is_reproducible_and_metadata_is_synthetic(self):
        self.assertEqual(generate_synthetic_responses(seed=42), (self.rows, self.registry, self.metadata))
        self.assertNotEqual(generate_synthetic_responses(seed=43)[0], self.rows)
        self.assertTrue(self.metadata["synthetic"])
        self.assertFalse(self.metadata["production_approved"])
        self.assertIn("SYNTHETIC", self.metadata["label"])
        self.assertEqual(self.metadata["seed"], 42)

    def test_each_required_field_is_checked(self):
        for field in self.rows[0]:
            with self.subTest(field=field):
                rows = copy.deepcopy(self.rows)
                del rows[0][field]
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_invalid_values_are_rejected(self):
        cases = [
            ("event_id", "not-a-uuid"), ("learner_id", "not-a-uuid"),
            ("topic_id", "Money Basics"), ("concept_id", "Money In"),
            ("question_id", "Question_1"), ("correct", 1), ("correct", "false"),
            ("answered_at", "2026-01-01T00:00:00"), ("answered_at", "not-a-date"),
            ("attempt_no", 0), ("attempt_no", True), ("attempt_no", 1.5),
            ("content_version", "synthetic-v1"), ("content_version", "g" * 64),
        ]
        for field, value in cases:
            with self.subTest(field=field, value=value):
                rows = copy.deepcopy(self.rows)
                rows[0][field] = value
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_unknown_fields_and_empty_input_are_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["name"] = "private"
        for invalid in [rows, [], None, [None]]:
            with self.subTest(invalid=type(invalid).__name__):
                with self.assertRaises(ValueError):
                    validate_answer_rows(invalid, self.registry)

    def test_obsolete_question_index_column_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["question_index"] = 0
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_invalid_or_ambiguous_registry_positions_are_rejected(self):
        key = ("money-basics", self.rows[1]["content_version"], "money-out")
        for position in [None, -1, True, 0]:
            registry = copy.deepcopy(self.registry)
            registry[key]["position"] = position
            with self.subTest(position=position), self.assertRaises(ValueError):
                validate_answer_rows(self.rows, registry)

    def test_postgres_integer_overflow_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["attempt_no"] = 2147483648
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_boolean_seed_is_not_accepted_as_integer(self):
        with self.assertRaises(ValueError):
            generate_synthetic_responses(seed=True)

    def test_duplicate_event_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["event_id"] = rows[0]["event_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_duplicate_question_within_submission_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["question_id"] = rows[0]["question_id"]
        rows[1]["concept_id"] = rows[0]["concept_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_unknown_version_question_or_concept_is_rejected(self):
        for field, value in [("content_version", "0" * 64), ("question_id", "unknown"),
                             ("concept_id", "money-out"), ("topic_id", "unknown")]:
            rows = copy.deepcopy(self.rows)
            rows[0][field] = value
            with self.subTest(field=field), self.assertRaises(ValueError):
                validate_answer_rows(rows, self.registry)

    def test_version_is_a_hash_of_the_synthetic_content(self):
        self.assertEqual(self.rows[0]["content_version"], self.metadata["content_version"])
        self.assertEqual(self.metadata["content_version"], synthetic_content_hash())

    def test_a_submission_cannot_mix_versions(self):
        rows = copy.deepcopy(self.rows)
        other_version = "1" * 64
        registry = dict(self.registry)
        registry[("money-basics", other_version, "money-out")] = {"concept_id": "money-out", "position": 1}
        rows[1]["content_version"] = other_version
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, registry)

    def test_pinned_content_orders_equal_timestamps_without_uuid_tiebreak(self):
        rows = list(reversed(self.rows))
        before = copy.deepcopy(rows)
        result = validate_answer_rows(rows, self.registry)
        self.assertEqual(rows, before)
        for left, right in zip(result, result[1:]):
            key = lambda r: (r["learner_id"], r["topic_id"], r["attempt_no"],
                             self.registry[(r["topic_id"], r["content_version"], r["question_id"])]["position"])
            self.assertLess(key(left), key(right))

    def test_timestamp_is_normalized_to_utc(self):
        rows = copy.deepcopy(self.rows)
        for row in rows:
            row["answered_at"] = row["answered_at"].replace("+00:00", "Z")
        self.assertTrue(all(row["answered_at"].endswith("+00:00")
                            for row in validate_answer_rows(rows, self.registry)))

    def test_time_cannot_go_backwards_in_attempt_order(self):
        rows = copy.deepcopy(self.rows)
        rows[2]["answered_at"] = "2025-01-01T00:00:00+00:00"
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_repeated_questions_across_attempts_are_valid(self):
        self.assertEqual(self.rows[0]["question_id"], self.rows[2]["question_id"])
        self.assertNotEqual(self.rows[0]["attempt_no"], self.rows[2]["attempt_no"])
        validate_answer_rows(self.rows, self.registry)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(SyntheticInputTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise AssertionError('SYNTHETIC input validation tests failed')
print('SYNTHETIC input checks passed. No training performed.')


In [ ]:
rows, registry, dataset_metadata = generate_synthetic_responses(seed=42)
validated_rows = validate_answer_rows(rows, registry)
print(LABEL)
print(json.dumps(dataset_metadata, indent=2, ensure_ascii=False))
print(f"SYNTHETIC: {len(validated_rows)} answer events, 12 fictional learners")
print(json.dumps(validated_rows[:2], indent=2))


## Increment 2: fit BKT on synthetic learners

Each concept gets four parameters: P(init) knows it before the first question, P(learn) learns it between attempts, P(slip) answers wrong despite knowing it, P(guess) answers right without knowing it. No forgetting; guess and slip are bounded so knowing stays distinguishable from not knowing.

Synthetic learners are drawn from known parameters, so the fit can be checked for **recovery** (true vs fitted) and for held-out **next-answer prediction** against always predicting the training correct rate. Learners are split by hash, never by row, so no learner is in both sets. These numbers say the fitter works, not how well real students are modelled.

In [ ]:
import math

# Standard BKT bounds: no forgetting, and guess + slip < 1 so "knows it" stays
# distinguishable from "doesn't". Guess allows up to 0.5 for two-option questions.
BKT_BOUNDS = {"p_init": (0.01, 0.99), "p_learn": (0.01, 0.5),
              "p_slip": (0.01, 0.3), "p_guess": (0.01, 0.5)}
# Hidden parameters the synthetic learners are simulated from, so the fit can be
# checked for recovery. Invented for engineering; not estimates of real students.
SYNTHETIC_TRUE_PARAMS = {
    "money-in": {"p_init": 0.25, "p_learn": 0.20, "p_slip": 0.08, "p_guess": 0.25},
    "money-out": {"p_init": 0.15, "p_learn": 0.12, "p_slip": 0.10, "p_guess": 0.20},
}
MASTERY_THRESHOLD = 0.95


def _check_params(params):
    if not isinstance(params, dict) or set(params) != set(BKT_BOUNDS):
        raise ValueError("expected exactly p_init, p_learn, p_slip and p_guess")
    for name, (low, high) in BKT_BOUNDS.items():
        value = params[name]
        if type(value) not in (int, float) or not low <= value <= high:
            raise ValueError(f"{name} must be between {low} and {high}")


def simulate_bkt_learners(true_params=None, n_learners=400, n_attempts=6, seed=7):
    """Rows in the nine-field contract, drawn from a known BKT process. SYNTHETIC."""
    true_params = SYNTHETIC_TRUE_PARAMS if true_params is None else true_params
    if type(seed) is not int or type(n_learners) is not int or type(n_attempts) is not int:
        raise ValueError("seed, n_learners and n_attempts must be integers")
    if n_learners < 1 or n_attempts < 1:
        raise ValueError("need at least one learner and one attempt")
    quiz = SYNTHETIC_CONTENT["variants"][0]["questions"]
    if set(true_params) != {q["conceptId"] for q in quiz}:
        raise ValueError("true_params must cover exactly the fixture's concepts")
    for params in true_params.values():
        _check_params(params)
    rng = random.Random(seed)
    version = synthetic_content_hash()
    registry = {("money-basics", version, q["id"]): {"concept_id": q["conceptId"], "position": position}
                for position, q in enumerate(quiz)}
    rows = []
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    for learner_index in range(n_learners):
        learner = str(UUID(int=rng.getrandbits(128), version=4))
        knows = {c: rng.random() < p["p_init"] for c, p in true_params.items()}
        for attempt in range(1, n_attempts + 1):
            stamp = (start + timedelta(days=attempt - 1, seconds=learner_index)).isoformat()
            for question in quiz:
                p = true_params[question["conceptId"]]
                known = knows[question["conceptId"]]
                correct = rng.random() < ((1 - p["p_slip"]) if known else p["p_guess"])
                rows.append({
                    "event_id": str(UUID(int=rng.getrandbits(128), version=4)),
                    "learner_id": learner, "topic_id": "money-basics",
                    "concept_id": question["conceptId"], "question_id": question["id"],
                    "correct": correct, "answered_at": stamp,
                    "attempt_no": attempt, "content_version": version,
                })
                if not known and rng.random() < p["p_learn"]:
                    knows[question["conceptId"]] = True
    metadata = {"label": LABEL, "synthetic": True, "production_approved": False,
                "seed": seed, "learners": n_learners, "attempts": n_attempts,
                "content_version": version, "true_params": true_params,
                "generator": "known BKT process; tests parameter recovery, not real learning"}
    return rows, registry, metadata


def build_sequences(validated_rows):
    """{concept_id: {learner_id: [correct, ...]}} in attempt and pinned-question order.
    Expects the output of validate_answer_rows, which is already sorted that way."""
    sequences = {}
    for row in validated_rows:
        sequences.setdefault(row["concept_id"], {}).setdefault(row["learner_id"], []).append(row["correct"])
    return sequences


def _emission(params, correct):
    """P(observation | unknown), P(observation | known)."""
    if correct:
        return params["p_guess"], 1 - params["p_slip"]
    return 1 - params["p_guess"], params["p_slip"]


def _forward_backward(params, sequence):
    learn = params["p_learn"]
    emissions = [_emission(params, o) for o in sequence]
    alphas, scales = [], []
    alpha = ((1 - params["p_init"]) * emissions[0][0], params["p_init"] * emissions[0][1])
    for t in range(len(sequence)):
        if t:
            prev = alphas[-1]
            alpha = (prev[0] * (1 - learn) * emissions[t][0],
                     (prev[0] * learn + prev[1]) * emissions[t][1])
        scale = alpha[0] + alpha[1]
        alphas.append((alpha[0] / scale, alpha[1] / scale))
        scales.append(scale)
    betas = [(1.0, 1.0)] * len(sequence)
    for t in range(len(sequence) - 2, -1, -1):
        e, b = emissions[t + 1], betas[t + 1]
        betas[t] = (((1 - learn) * e[0] * b[0] + learn * e[1] * b[1]) / scales[t + 1],
                    e[1] * b[1] / scales[t + 1])
    return alphas, betas, emissions, sum(math.log(s) for s in scales)


def _clip(params):
    return {k: min(max(v, BKT_BOUNDS[k][0]), BKT_BOUNDS[k][1]) for k, v in params.items()}


def _em(sequences, params, iterations, tolerance):
    previous = -math.inf
    for _ in range(iterations):
        init = known_total = unknown_total = slips = guesses = learned = unknown_before = 0.0
        log_likelihood = 0.0
        for sequence in sequences:
            alphas, betas, emissions, ll = _forward_backward(params, sequence)
            log_likelihood += ll
            gammas = []
            for a, b in zip(alphas, betas):
                g0, g1 = a[0] * b[0], a[1] * b[1]
                gammas.append((g0 / (g0 + g1), g1 / (g0 + g1)))
            init += gammas[0][1]
            for t, (g, observed) in enumerate(zip(gammas, sequence)):
                unknown_total += g[0]
                known_total += g[1]
                slips += g[1] * (not observed)
                guesses += g[0] * observed
                if t + 1 < len(sequence):
                    e, b = emissions[t + 1], betas[t + 1]
                    stay = alphas[t][0] * (1 - params["p_learn"]) * e[0] * b[0]
                    learn = alphas[t][0] * params["p_learn"] * e[1] * b[1]
                    kept = alphas[t][1] * e[1] * b[1]
                    learned += learn / (stay + learn + kept)
                    unknown_before += g[0]
        params = _clip({
            "p_init": init / len(sequences),
            "p_learn": learned / unknown_before if unknown_before else params["p_learn"],
            "p_slip": slips / known_total if known_total else params["p_slip"],
            "p_guess": guesses / unknown_total if unknown_total else params["p_guess"],
        })
        if log_likelihood - previous < tolerance:
            break
        previous = log_likelihood
    return params, _forward_backward_total(params, sequences)


def _forward_backward_total(params, sequences):
    return sum(_forward_backward(params, s)[3] for s in sequences)


def fit_bkt(sequences, iterations=200, tolerance=1e-6):
    """Fit one concept's BKT parameters by EM (Baum-Welch) from several starting points.
    `sequences` is a list of per-learner lists of booleans. Returns (params, log-likelihood)."""
    if not isinstance(sequences, list) or not sequences:
        raise ValueError("expected a non-empty list of answer sequences")
    for sequence in sequences:
        if not isinstance(sequence, list) or not sequence or any(type(o) is not bool for o in sequence):
            raise ValueError("each sequence must be a non-empty list of booleans")
    starts = [
        {"p_init": 0.3, "p_learn": 0.1, "p_slip": 0.1, "p_guess": 0.2},
        {"p_init": 0.1, "p_learn": 0.3, "p_slip": 0.05, "p_guess": 0.3},
        {"p_init": 0.5, "p_learn": 0.05, "p_slip": 0.2, "p_guess": 0.1},
    ]
    return max((_em(sequences, start, iterations, tolerance) for start in starts), key=lambda fit: fit[1])


def predict_sequence(params, sequence):
    """One-step-ahead P(correct) before each answer, and P(known) after the last one."""
    _check_params(params)
    known, predictions = params["p_init"], []
    for observed in sequence:
        if type(observed) is not bool:
            raise ValueError("observations must be booleans")
        p_correct = known * (1 - params["p_slip"]) + (1 - known) * params["p_guess"]
        predictions.append(p_correct)
        posterior = (known * (1 - params["p_slip"]) / p_correct if observed
                     else known * params["p_slip"] / (1 - p_correct))
        known = posterior + (1 - posterior) * params["p_learn"]
    return predictions, known


def split_learners(learner_ids, test_share=0.2):
    """Deterministic learner-level split by SHA-256, so no learner is in both sets."""
    if not 0 < test_share < 1:
        raise ValueError("test_share must be between 0 and 1")
    train, test = [], []
    for learner in sorted(set(learner_ids)):
        bucket = int(hashlib.sha256(learner.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
        (test if bucket < test_share else train).append(learner)
    if not train or not test:
        raise ValueError("split left one side empty; need more learners")
    return train, test


def _auc(labels, scores):
    positives = [s for l, s in zip(labels, scores) if l]
    negatives = [s for l, s in zip(labels, scores) if not l]
    if not positives or not negatives:
        return None
    wins = sum(1.0 if p > n else 0.5 if p == n else 0.0 for p in positives for n in negatives)
    return wins / (len(positives) * len(negatives))


def evaluate(params, sequences, baseline_rate):
    """Held-out next-answer metrics for BKT against a constant training-set rate."""
    labels, bkt, mastered = [], [], 0
    for sequence in sequences:
        predictions, final_known = predict_sequence(params, sequence)
        labels.extend(sequence)
        bkt.extend(predictions)
        mastered += final_known >= MASTERY_THRESHOLD

    def log_loss(scores):
        eps = 1e-9
        return -sum(math.log(max(eps, s if l else 1 - s)) for l, s in zip(labels, scores)) / len(labels)

    constant = [baseline_rate] * len(labels)
    return {
        "answers": len(labels),
        "bkt_log_loss": round(log_loss(bkt), 4),
        "baseline_log_loss": round(log_loss(constant), 4),
        "bkt_auc": None if _auc(labels, bkt) is None else round(_auc(labels, bkt), 4),
        "bkt_accuracy": round(sum((s >= 0.5) == l for l, s in zip(labels, bkt)) / len(labels), 4),
        "baseline_accuracy": round(sum((baseline_rate >= 0.5) == l for l in labels) / len(labels), 4),
        "share_mastered_at_end": round(mastered / len(sequences), 4),
    }


def export_draft_params(fits, evaluations, dataset_metadata):
    """Draft artifact for review. Synthetic: never load it as the app's mastery model."""
    return {
        "schemaVersion": 1,
        "model": "bkt",
        "label": LABEL,
        "synthetic": True,
        "productionApproved": False,
        "notForAppUse": "Fitted on synthetic learners; app priors and unlock rules are unchanged.",
        "source": {"seed": dataset_metadata["seed"], "learners": dataset_metadata["learners"],
                   "attempts": dataset_metadata["attempts"],
                   "contentVersion": dataset_metadata["content_version"]},
        "masteryThreshold": MASTERY_THRESHOLD,
        "concepts": {
            concept: {"pInit": round(p["p_init"], 4), "pLearn": round(p["p_learn"], 4),
                      "pSlip": round(p["p_slip"], 4), "pGuess": round(p["p_guess"], 4)}
            for concept, p in sorted(fits.items())
        },
        "heldOutEvaluation": evaluations,
    }

In [ ]:
import unittest


class BktFitTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        rows, cls.registry, cls.metadata = simulate_bkt_learners(n_learners=300, n_attempts=6, seed=11)
        cls.rows = validate_answer_rows(rows, cls.registry)
        cls.sequences = build_sequences(cls.rows)

    def test_simulated_rows_pass_the_input_contract(self):
        self.assertEqual(len(self.rows), 300 * 6 * 2)
        self.assertTrue(self.metadata["synthetic"])
        self.assertFalse(self.metadata["production_approved"])

    def test_sequences_follow_attempt_order(self):
        self.assertEqual(set(self.sequences), {"money-in", "money-out"})
        self.assertTrue(all(len(s) == 6 for c in self.sequences.values() for s in c.values()))

    def test_fit_recovers_known_parameters(self):
        for concept, truth in SYNTHETIC_TRUE_PARAMS.items():
            fitted, log_likelihood = fit_bkt(list(self.sequences[concept].values()))
            self.assertTrue(math.isfinite(log_likelihood))
            for name, value in truth.items():
                with self.subTest(concept=concept, param=name):
                    self.assertAlmostEqual(fitted[name], value, delta=0.08)

    def test_fit_stays_in_bounds_when_everyone_is_right(self):
        fitted, _ = fit_bkt([[True, True, True]] * 20)
        for name, (low, high) in BKT_BOUNDS.items():
            self.assertTrue(low <= fitted[name] <= high, name)

    def test_fit_handles_single_answers(self):
        fitted, _ = fit_bkt([[True], [False], [True]])
        self.assertEqual(set(fitted), set(BKT_BOUNDS))

    def test_fit_rejects_invalid_input(self):
        for invalid in [[], None, [[]], [[1, 0]], [["true"]], "TFT"]:
            with self.subTest(invalid=invalid), self.assertRaises(ValueError):
                fit_bkt(invalid)

    def test_prediction_rises_after_correct_answers(self):
        params = SYNTHETIC_TRUE_PARAMS["money-in"]
        predictions, known = predict_sequence(params, [True, True, True])
        self.assertLess(predictions[0], predictions[1])
        self.assertLess(predictions[1], predictions[2])
        self.assertGreater(known, params["p_init"])

    def test_prediction_rejects_invalid_params_and_observations(self):
        params = dict(SYNTHETIC_TRUE_PARAMS["money-in"])
        for bad in [{**params, "p_slip": 0.9}, {**params, "p_guess": True}, {"p_init": 0.5}]:
            with self.subTest(bad=bad), self.assertRaises(ValueError):
                predict_sequence(bad, [True])
        with self.assertRaises(ValueError):
            predict_sequence(params, [1])

    def test_split_keeps_learners_apart_and_is_deterministic(self):
        learners = [r["learner_id"] for r in self.rows]
        train, test = split_learners(learners)
        self.assertFalse(set(train) & set(test))
        self.assertEqual(len(train) + len(test), 300)
        self.assertEqual((train, test), split_learners(list(reversed(learners))))
        with self.assertRaises(ValueError):
            split_learners(learners, test_share=1)

    def test_held_out_bkt_beats_constant_baseline(self):
        train, test = split_learners([r["learner_id"] for r in self.rows])
        for concept, by_learner in self.sequences.items():
            train_seqs = [by_learner[l] for l in train]
            rate = sum(map(sum, train_seqs)) / sum(map(len, train_seqs))
            fitted, _ = fit_bkt(train_seqs)
            result = evaluate(fitted, [by_learner[l] for l in test], rate)
            with self.subTest(concept=concept):
                self.assertLess(result["bkt_log_loss"], result["baseline_log_loss"])
                self.assertGreater(result["bkt_auc"], 0.5)

    def test_export_is_labelled_synthetic_and_not_approved(self):
        fits = {c: fit_bkt(list(s.values()))[0] for c, s in self.sequences.items()}
        draft = export_draft_params(fits, {}, self.metadata)
        self.assertTrue(draft["synthetic"])
        self.assertFalse(draft["productionApproved"])
        self.assertIn("SYNTHETIC", draft["label"])
        self.assertEqual(set(draft["concepts"]), {"money-in", "money-out"})
        json.dumps(draft)

    def test_simulation_rejects_invalid_arguments(self):
        for kwargs in [{"seed": True}, {"n_learners": 0}, {"n_attempts": 1.5},
                       {"true_params": {"money-in": SYNTHETIC_TRUE_PARAMS["money-in"]}}]:
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                simulate_bkt_learners(**kwargs)


# Runs in the notebook; test_bkt_fit.py loads this class without running it here.
if __name__ == "__main__":
    fit_result = unittest.TextTestRunner(verbosity=1).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(BktFitTests))
    if not fit_result.wasSuccessful():
        raise AssertionError("SYNTHETIC BKT fitting tests failed")
    print("SYNTHETIC BKT fitting checks passed.")

In [ ]:
sim_rows, sim_registry, sim_metadata = simulate_bkt_learners(n_learners=400, n_attempts=6, seed=7)
sim_validated = validate_answer_rows(sim_rows, sim_registry)
sim_sequences = build_sequences(sim_validated)
train_ids, test_ids = split_learners([row["learner_id"] for row in sim_validated])
print(LABEL)
print(f"SYNTHETIC: {len(sim_validated)} answers; {len(train_ids)} train / {len(test_ids)} held-out learners")

fits, evaluations = {}, {}
for concept, by_learner in sorted(sim_sequences.items()):
    train_seqs = [by_learner[learner] for learner in train_ids]
    rate = sum(map(sum, train_seqs)) / sum(map(len, train_seqs))
    fits[concept], _ = fit_bkt(train_seqs)
    evaluations[concept] = evaluate(fits[concept], [by_learner[learner] for learner in test_ids], rate)
    print(f"\n{concept}")
    print(f"  {'param':<8} {'true':>6} {'fitted':>7}")
    for name in BKT_BOUNDS:
        print(f"  {name:<8} {SYNTHETIC_TRUE_PARAMS[concept][name]:>6.3f} {fits[concept][name]:>7.3f}")
    print("  held-out:", json.dumps(evaluations[concept]))

draft = export_draft_params(fits, evaluations, sim_metadata)
path = "knowledge-tracing-params.synthetic-draft.json"
with open(path, "w", encoding="utf-8") as handle:
    json.dump(draft, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
print(f"\nWrote {path} (SYNTHETIC draft; do not load it in the app).")
try:
    from google.colab import files
    files.download(path)
except ImportError:
    pass

## Increment 3: real-data benchmark on ASSISTments 2009–10

**Data:** ASSISTments 2009–2010 skill builder data (corrected), real answers from about 4,200 US middle-school maths students, https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010. Please also cite Feng, M., Heffernan, N.T. & Koedinger, K.R. (2009), *Addressing the assessment challenge in an Intelligent Tutoring System that tutors as it assesses*, User Modeling and User-Adapted Interaction 19, 243–266.

**Terms (accepted by R1 on 2026-10-11):** the file is downloaded at run time and **never committed or shared**; our code stays public; no attempt to identify students. Only aggregate results leave this notebook.

**Models**, all scored on the same held-out answers (every held-out learner's answers after their first):
- *Overall / per-skill correct rate*: baselines.
- *BKT*: the model above, vectorised (tests check it matches the reviewed Python version).
- *BKT + forgetting*: adds P(forget), a known skill can be lost between answers.
- *PFA* (Performance Factors Analysis): per-skill logistic regression on earlier successes and failures.
- *DKT* (Deep Knowledge Tracing, Piech et al. 2015): an LSTM over each learner's whole history, trained on the GPU with early stopping.

Rows without a skill are dropped; multi-skill problems (`1_13`) count as one combined skill. These are maths skills: what carries over to Gullak is the model choice and realistic parameter ranges, not the parameters themselves.

In [ ]:
import csv
import os
import urllib.request

import numpy as np

ASSISTMENTS_PAGE = ("https://sites.google.com/site/assistmentsdata/home/"
                    "2009-2010-assistment-data/skill-builder-data-2009-2010")
ASSISTMENTS_URL = ("https://drive.usercontent.google.com/download?"
                   "id=1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE&export=download&confirm=t")
ASSISTMENTS_CITATION = (
    "ASSISTments 2009-2010 skill builder data (corrected), " + ASSISTMENTS_PAGE
    + ". Feng, M., Heffernan, N.T. & Koedinger, K.R. (2009). Addressing the assessment challenge"
    " in an Intelligent Tutoring System that tutors as it assesses. User Modeling and"
    " User-Adapted Interaction, 19, 243-266.")
FORGET_BOUNDS = (0.0, 0.3)
BATCH_STARTS = [
    {"p_init": 0.3, "p_learn": 0.1, "p_slip": 0.1, "p_guess": 0.2},
    {"p_init": 0.6, "p_learn": 0.2, "p_slip": 0.05, "p_guess": 0.3},
]


def download_assistments(path="assistments-2009-2010-skill-builder.csv"):
    """Fetch the CSV at run time. The terms forbid redistributing it: never commit it."""
    if not os.path.exists(path):
        urllib.request.urlretrieve(ASSISTMENTS_URL, path)
    with open(path, "rb") as handle:
        if b"order_id" not in handle.read(400):
            os.remove(path)
            raise ValueError("download was not the ASSISTments CSV; Drive may be rate-limiting, retry later")
    return path


def load_assistments(lines):
    """Answers in canonical order: by learner, then order_id (time).
    Rows without a skill are dropped; multi-skill problems ("1_13") stay one combined skill."""
    reader = csv.DictReader(lines)
    missing = {"order_id", "user_id", "skill_id", "correct"} - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")
    records, dropped, seen = [], 0, set()
    for row in reader:
        skill = (row["skill_id"] or "").strip()
        if skill in ("", "NA"):
            dropped += 1
            continue
        if row["correct"] not in ("0", "1"):
            raise ValueError(f"order {row['order_id']}: correct must be 0 or 1")
        order = int(row["order_id"])
        if order in seen:
            raise ValueError(f"duplicate order_id {order}")
        seen.add(order)
        records.append((row["user_id"], order, skill, row["correct"] == "1"))
    if not records:
        raise ValueError("no answers with a skill")
    records.sort(key=lambda r: (r[0], r[1]))
    learners = sorted({r[0] for r in records})
    skills = sorted({r[2] for r in records})
    learner_index = {u: i for i, u in enumerate(learners)}
    skill_index = {k: i for i, k in enumerate(skills)}
    learner = np.array([learner_index[r[0]] for r in records])
    starts = np.r_[0, np.flatnonzero(np.diff(learner)) + 1]
    position = np.arange(len(records)) - np.repeat(starts, np.diff(np.r_[starts, len(records)]))
    return {
        "learner": learner,
        "skill": np.array([skill_index[r[2]] for r in records]),
        "correct": np.array([r[3] for r in records], dtype=float),
        "position": position,
        "learners": learners, "skills": skills, "dropped_without_skill": dropped,
    }


def learner_folds(learners, folds=5):
    """Fold per learner from a SHA-256 of the id: deterministic, never splits a learner."""
    if type(folds) is not int or folds < 2:
        raise ValueError("folds must be an integer >= 2")
    return np.array([int(hashlib.sha256(str(u).encode("utf-8")).hexdigest()[:8], 16) % folds
                     for u in learners])


def _sequences(data, mask):
    """Per (learner, skill) answer sequences, longest first, as columns of answer indices."""
    idx = np.flatnonzero(mask)
    order = idx[np.lexsort((data["position"][idx], data["skill"][idx], data["learner"][idx]))]
    learner, skill = data["learner"][order], data["skill"][order]
    change = np.r_[True, (learner[1:] != learner[:-1]) | (skill[1:] != skill[:-1])]
    starts = np.flatnonzero(change)
    lengths = np.diff(np.r_[starts, len(order)])
    by_length = np.argsort(-lengths, kind="stable")
    starts, lengths = starts[by_length], lengths[by_length]
    active = [int(np.count_nonzero(lengths > t)) for t in range(int(lengths.max()))]
    return {"skill": skill[starts], "active": active,
            "cols": [order[starts[:n] + t] for t, n in enumerate(active)]}


def _gather(params, seqs):
    return {name: values[seqs["skill"]] for name, values in params.items()}


def _bkt_pass(params, seqs, y):
    """Scaled forward-backward over every sequence at once."""
    p = _gather(params, seqs)
    learn, forget, slip, guess = p["p_learn"], p["p_forget"], p["p_slip"], p["p_guess"]
    emissions, alphas, scales = [], [], []
    for t, cols in enumerate(seqs["cols"]):
        n, o = seqs["active"][t], y[cols]
        e = np.stack([np.where(o == 1, guess[:n], 1 - guess[:n]),
                      np.where(o == 1, 1 - slip[:n], slip[:n])], axis=1)
        if t == 0:
            a = np.stack([1 - p["p_init"], p["p_init"]], axis=1) * e
        else:
            prev = alphas[-1][:n]
            a = np.stack([prev[:, 0] * (1 - learn[:n]) + prev[:, 1] * forget[:n],
                          prev[:, 0] * learn[:n] + prev[:, 1] * (1 - forget[:n])], axis=1) * e
        c = a.sum(axis=1)
        emissions.append(e)
        alphas.append(a / c[:, None])
        scales.append(c)
    betas = [None] * len(alphas)
    betas[-1] = np.ones((seqs["active"][-1], 2))
    for t in range(len(alphas) - 2, -1, -1):
        m = seqs["active"][t + 1]
        b = np.ones((seqs["active"][t], 2))
        v = emissions[t + 1] * betas[t + 1] / scales[t + 1][:, None]
        b[:m, 0] = (1 - learn[:m]) * v[:, 0] + learn[:m] * v[:, 1]
        b[:m, 1] = forget[:m] * v[:, 0] + (1 - forget[:m]) * v[:, 1]
        betas[t] = b
    return emissions, alphas, betas, scales


def _skill_log_likelihood(seqs, scales, n_skills):
    return sum(np.bincount(seqs["skill"][:len(c)], np.log(c), n_skills) for c in scales)


def _m_step(params, seqs, y, passed, forgetting):
    emissions, alphas, betas, scales = passed
    k = len(params["p_init"])
    s = seqs["skill"]
    sums = {name: np.zeros(k) for name in
            ("init", "unknown", "known", "slips", "guesses", "learned", "forgot", "unknown_before", "known_before")}
    p = _gather(params, seqs)
    for t, cols in enumerate(seqs["cols"]):
        n, o = seqs["active"][t], y[cols]
        g = alphas[t] * betas[t]
        g /= g.sum(axis=1, keepdims=True)
        if t == 0:
            sums["init"] += np.bincount(s, g[:, 1], k)
        sums["unknown"] += np.bincount(s[:n], g[:, 0], k)
        sums["known"] += np.bincount(s[:n], g[:, 1], k)
        sums["slips"] += np.bincount(s[:n], g[:, 1] * (1 - o), k)
        sums["guesses"] += np.bincount(s[:n], g[:, 0] * o, k)
        if t + 1 < len(seqs["cols"]):
            m = seqs["active"][t + 1]
            v = emissions[t + 1] * betas[t + 1] / scales[t + 1][:, None]
            a = alphas[t][:m]
            stay = a[:, 0] * (1 - p["p_learn"][:m]) * v[:, 0]
            learn = a[:, 0] * p["p_learn"][:m] * v[:, 1]
            forget = a[:, 1] * p["p_forget"][:m] * v[:, 0]
            keep = a[:, 1] * (1 - p["p_forget"][:m]) * v[:, 1]
            total = stay + learn + forget + keep
            sums["learned"] += np.bincount(s[:m], learn / total, k)
            sums["forgot"] += np.bincount(s[:m], forget / total, k)
            sums["unknown_before"] += np.bincount(s[:m], g[:m, 0], k)
            sums["known_before"] += np.bincount(s[:m], g[:m, 1], k)

    def ratio(numerator, denominator, old):
        out = old.copy()
        np.divide(numerator, denominator, out=out, where=denominator > 0)
        return out

    sequences = np.bincount(s, minlength=k).astype(float)
    new = {
        "p_init": ratio(sums["init"], sequences, params["p_init"]),
        "p_learn": ratio(sums["learned"], sums["unknown_before"], params["p_learn"]),
        "p_slip": ratio(sums["slips"], sums["known"], params["p_slip"]),
        "p_guess": ratio(sums["guesses"], sums["unknown"], params["p_guess"]),
    }
    new = {name: np.clip(v, *BKT_BOUNDS[name]) for name, v in new.items()}
    new["p_forget"] = (np.clip(ratio(sums["forgot"], sums["known_before"], params["p_forget"]), *FORGET_BOUNDS)
                       if forgetting else np.zeros(k))
    return new


def fit_bkt_batch(data, mask, forgetting=False, iterations=100, tolerance=1e-7, starts=None):
    """Per-skill BKT by EM, vectorised over all sequences; best start chosen per skill.
    tolerance is relative to the total log-likelihood."""
    seqs, y, k = _sequences(data, mask), data["correct"], len(data["skills"])
    best, best_ll = None, None
    for start in starts or BATCH_STARTS:
        params = {name: np.full(k, float(value)) for name, value in start.items()}
        params["p_forget"] = np.full(k, 0.05 if forgetting else 0.0)
        previous = -np.inf
        for _ in range(iterations):
            passed = _bkt_pass(params, seqs, y)
            log_likelihood = sum(np.log(c).sum() for c in passed[3])
            params = _m_step(params, seqs, y, passed, forgetting)
            if log_likelihood - previous < tolerance * abs(log_likelihood):
                break
            previous = log_likelihood
        skill_ll = _skill_log_likelihood(seqs, _bkt_pass(params, seqs, y)[3], k)
        if best is None:
            best, best_ll = params, skill_ll
        else:
            better = skill_ll > best_ll
            best = {name: np.where(better, params[name], best[name]) for name in params}
            best_ll = np.maximum(skill_ll, best_ll)
    return best


def predict_bkt_batch(params, data, mask):
    """One-step-ahead P(correct) for every masked answer (NaN elsewhere)."""
    seqs, y = _sequences(data, mask), data["correct"]
    p = _gather(params, seqs)
    out = np.full(len(y), np.nan)
    known = p["p_init"].copy()
    for t, cols in enumerate(seqs["cols"]):
        n = seqs["active"][t]
        known, slip, guess = known[:n], p["p_slip"][:n], p["p_guess"][:n]
        predicted = known * (1 - slip) + (1 - known) * guess
        out[cols] = predicted
        posterior = np.where(y[cols] == 1, known * (1 - slip) / predicted, known * slip / (1 - predicted))
        known = posterior * (1 - p["p_forget"][:n]) + (1 - posterior) * p["p_learn"][:n]
    return out


def pfa_features(data):
    """Earlier successes and failures of the same learner on the same skill."""
    order = np.lexsort((data["position"], data["skill"], data["learner"]))
    learner, skill, y = data["learner"][order], data["skill"][order], data["correct"][order]
    start = np.r_[True, (learner[1:] != learner[:-1]) | (skill[1:] != skill[:-1])]
    group = np.cumsum(start) - 1
    first = np.flatnonzero(start)
    seen = np.arange(len(order)) - first[group]
    cumulative = np.cumsum(y) - y
    successes = cumulative - cumulative[first][group]
    out_s, out_f = np.empty(len(order)), np.empty(len(order))
    out_s[order], out_f[order] = successes, seen - successes
    return out_s, out_f


def _sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


def fit_pfa(data, mask, iterations=30, ridge=1.0):
    """Performance Factors Analysis: per-skill logistic regression on prior successes and
    failures, fitted by Newton's method with a small ridge penalty."""
    successes, failures = pfa_features(data)
    x = np.stack([np.ones(mask.sum()), successes[mask], failures[mask]], axis=1)
    k, s, y = len(data["skills"]), data["skill"][mask], data["correct"][mask]
    theta = np.zeros((k, 3))
    for _ in range(iterations):
        p = _sigmoid((x * theta[s]).sum(axis=1))
        weight = p * (1 - p)
        gradient = np.stack([np.bincount(s, x[:, j] * (y - p), k) for j in range(3)], axis=1) - ridge * theta
        hessian = np.empty((k, 3, 3))
        for i in range(3):
            for j in range(3):
                hessian[:, i, j] = -np.bincount(s, weight * x[:, i] * x[:, j], k)
        hessian -= ridge * np.eye(3)
        step = np.linalg.solve(hessian, gradient[:, :, None])[:, :, 0]
        theta -= step
        if np.abs(step).max() < 1e-8:
            break
    return theta


def predict_pfa(theta, data, mask):
    successes, failures = pfa_features(data)
    out = np.full(len(data["correct"]), np.nan)
    s = data["skill"][mask]
    out[mask] = _sigmoid(theta[s, 0] + theta[s, 1] * successes[mask] + theta[s, 2] * failures[mask])
    return out


def auc(labels, scores):
    """Area under the ROC curve with tied scores sharing their average rank."""
    labels, scores = np.asarray(labels, dtype=float), np.asarray(scores, dtype=float)
    positives = labels.sum()
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return float("nan")
    unique, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)
    upper = np.cumsum(counts)
    ranks = (upper - (counts - 1) / 2)[inverse]
    return float((ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def score(labels, predictions):
    if np.isnan(predictions).any():
        raise ValueError("missing predictions")
    clipped = np.clip(predictions, 1e-7, 1 - 1e-7)
    return {
        "auc": auc(labels, predictions),
        "rmse": float(np.sqrt(np.mean((labels - predictions) ** 2))),
        "accuracy": float(np.mean((predictions >= 0.5) == (labels == 1))),
        "log_loss": float(-np.mean(labels * np.log(clipped) + (1 - labels) * np.log(1 - clipped))),
    }


def evaluation_mask(data, test):
    """Held-out answers except each learner's first, which no sequence model can predict."""
    return test & (data["position"] > 0)


def run_classic_benchmark(data, folds=5, log=print):
    """5-fold, learner-separated comparison of baselines, BKT, BKT + forgetting and PFA."""
    fold_of = learner_folds(data["learners"], folds)[data["learner"]]
    y, k = data["correct"], len(data["skills"])
    results, bkt_fits = {}, []
    for fold in range(folds):
        train, test = fold_of != fold, fold_of == fold
        evaluate_on = evaluation_mask(data, test)
        rate = y[train].mean()
        skill_rate = np.where(np.bincount(data["skill"][train], minlength=k) > 0,
                              np.bincount(data["skill"][train], y[train], k)
                              / np.maximum(np.bincount(data["skill"][train], minlength=k), 1), rate)
        bkt = fit_bkt_batch(data, train)
        predictions = {
            "Overall correct rate": np.full(len(y), rate),
            "Per-skill correct rate": skill_rate[data["skill"]],
            "BKT": predict_bkt_batch(bkt, data, test),
            "BKT + forgetting": predict_bkt_batch(fit_bkt_batch(data, train, forgetting=True), data, test),
            "PFA": predict_pfa(fit_pfa(data, train), data, test),
        }
        for name, predicted in predictions.items():
            results.setdefault(name, []).append(score(y[evaluate_on], predicted[evaluate_on]))
        bkt_fits.append((bkt, np.bincount(data["skill"][train], minlength=k)))
        log(f"fold {fold + 1}/{folds}: " + ", ".join(
            f"{name} AUC {results[name][-1]['auc']:.3f}" for name in predictions))
    return results, bkt_fits


def summarise(results):
    return {name: {metric: {"mean": round(float(np.mean([f[metric] for f in folds])), 4),
                            "std": round(float(np.std([f[metric] for f in folds])), 4)}
                   for metric in folds[0]}
            for name, folds in results.items()}


def parameter_ranges(bkt_fits, min_answers=200):
    """Spread of fitted BKT parameters over well-supported skills, pooled across folds.
    Used as starting ranges for Gullak concepts until our own data exists."""
    pooled = {name: [] for name in BKT_BOUNDS}
    for params, answers in bkt_fits:
        keep = answers >= min_answers
        for name in BKT_BOUNDS:
            pooled[name].extend(params[name][keep].tolist())
    return {name: {f"p{q}": round(float(np.percentile(values, q)), 4) for q in (10, 25, 50, 75, 90)}
            for name, values in pooled.items()} | {"skills_per_fold": int(np.mean(
                [np.count_nonzero(a >= min_answers) for _, a in bkt_fits]))}


def run_dkt(data, folds=5, hidden=128, epochs=30, patience=3, batch_size=32, lr=1e-3,
            seed=0, device=None, log=print):
    """Deep Knowledge Tracing (Piech et al., 2015): an LSTM over each learner's whole answer
    history predicts the next answer's correctness on its skill. Same folds and evaluation
    answers as the classic models; 10% of each training fold's learners pick the epoch."""
    import torch
    from torch import nn

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    k, y = len(data["skills"]), data["correct"]
    fold_of_learner = learner_folds(data["learners"], folds)
    validation_of_learner = learner_folds([f"validation:{u}" for u in data["learners"]], 10) == 0
    bounds = np.r_[0, np.flatnonzero(np.diff(data["learner"])) + 1, len(y)]
    spans = [(int(bounds[i]), int(bounds[i + 1])) for i in range(len(bounds) - 1)]

    class Dkt(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(2 * k, hidden)
            self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
            self.out = nn.Linear(hidden, k)

        def forward(self, tokens):
            return self.out(self.lstm(self.embed(tokens))[0])

    def batches(learners, shuffle, rng):
        learners = sorted(learners, key=lambda u: spans[u][1] - spans[u][0])
        groups = [learners[i:i + batch_size] for i in range(0, len(learners), batch_size)]
        if shuffle:
            rng.shuffle(groups)
        for group in groups:
            longest = max(spans[u][1] - spans[u][0] for u in group) - 1
            tokens = np.zeros((len(group), longest), dtype=np.int64)
            target_skill = np.zeros((len(group), longest), dtype=np.int64)
            target = np.zeros((len(group), longest), dtype=np.float32)
            mask = np.zeros((len(group), longest), dtype=bool)
            index = np.full((len(group), longest), -1, dtype=np.int64)
            for row, u in enumerate(group):
                start, end = spans[u]
                n = end - start - 1
                tokens[row, :n] = data["skill"][start:end - 1] * 2 + y[start:end - 1]
                target_skill[row, :n] = data["skill"][start + 1:end]
                target[row, :n] = y[start + 1:end]
                mask[row, :n] = True
                index[row, :n] = np.arange(start + 1, end)
            yield [torch.from_numpy(a).to(device) for a in (tokens, target_skill, target, mask)] + [index]

    def predict(model, learners):
        out = np.full(len(y), np.nan)
        model.eval()
        with torch.no_grad():
            for tokens, target_skill, _, mask, index in batches(learners, False, None):
                logits = model(tokens).gather(2, target_skill.unsqueeze(2)).squeeze(2)
                probabilities = torch.sigmoid(logits)[mask].cpu().numpy()
                out[index[mask.cpu().numpy()]] = probabilities
        return out

    results = []
    for fold in range(folds):
        torch.manual_seed(seed + fold)
        rng = np.random.default_rng(seed + fold)
        in_fold = [u for u in range(len(spans)) if spans[u][1] - spans[u][0] > 1]
        test = [u for u in in_fold if fold_of_learner[u] == fold]
        validation = [u for u in in_fold if fold_of_learner[u] != fold and validation_of_learner[u]]
        train = [u for u in in_fold if fold_of_learner[u] != fold and not validation_of_learner[u]]
        model = Dkt().to(device)
        optimiser = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()
        best_auc, best_state, waited = -1.0, None, 0
        validation_answers = np.zeros(len(y), dtype=bool)
        for u in validation:
            validation_answers[spans[u][0] + 1:spans[u][1]] = True
        for epoch in range(epochs):
            model.train()
            for tokens, target_skill, target, mask, _ in batches(train, True, rng):
                logits = model(tokens).gather(2, target_skill.unsqueeze(2)).squeeze(2)
                loss = loss_fn(logits[mask], target[mask])
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
            validation_auc = auc(y[validation_answers], predict(model, validation)[validation_answers])
            if validation_auc > best_auc + 1e-4:
                best_auc, waited = validation_auc, 0
                best_state = {name: t.detach().clone() for name, t in model.state_dict().items()}
            else:
                waited += 1
                if waited >= patience:
                    break
        model.load_state_dict(best_state)
        predicted = predict(model, test)
        test_mask = np.isin(data["learner"], test)
        evaluate_on = evaluation_mask(data, test_mask)
        results.append(score(y[evaluate_on], predicted[evaluate_on]))
        log(f"DKT fold {fold + 1}/{folds}: AUC {results[-1]['auc']:.3f} "
            f"(best epoch {epoch + 1 - waited}, validation AUC {best_auc:.3f}, {device})")
    return results

In [ ]:
import importlib.util
import io
import unittest


def fake_assistments_csv(n_learners=80, seed=3):
    """Small CSV in the ASSISTments column layout, from known BKT parameters. Test-only."""
    rng = random.Random(seed)
    truth = {"10": SYNTHETIC_TRUE_PARAMS["money-in"], "11": SYNTHETIC_TRUE_PARAMS["money-out"],
             "10_11": {"p_init": 0.4, "p_learn": 0.3, "p_slip": 0.05, "p_guess": 0.3}}
    out = io.StringIO()
    writer = csv.writer(out)
    writer.writerow(["", "order_id", "user_id", "skill_id", "skill_name", "correct", "original"])
    order = 1000
    for learner in range(n_learners):
        knows = {s: rng.random() < p["p_init"] for s, p in truth.items()}
        for _ in range(rng.randint(4, 14)):
            skill = rng.choice(list(truth) + [""])
            order += rng.randint(1, 5)
            if not skill:
                writer.writerow([order, order, 70000 + learner, "", "", 1, 1])
                continue
            p = truth[skill]
            correct = rng.random() < ((1 - p["p_slip"]) if knows[skill] else p["p_guess"])
            writer.writerow([order, order, 70000 + learner, skill, "fake", int(correct), 1])
            if not knows[skill] and rng.random() < p["p_learn"]:
                knows[skill] = True
    return out.getvalue()


class AssistmentsBenchmarkTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.text = fake_assistments_csv()
        cls.data = load_assistments(io.StringIO(cls.text))

    def test_loader_orders_by_learner_then_time_and_drops_rows_without_skill(self):
        d = self.data
        self.assertGreater(d["dropped_without_skill"], 0)
        self.assertEqual(d["skills"], ["10", "10_11", "11"])
        self.assertTrue(np.all(np.diff(d["learner"]) >= 0))
        self.assertTrue(np.all(d["position"][np.r_[True, np.diff(d["learner"]) != 0]] == 0))

    def test_loader_rejects_bad_input(self):
        header = "order_id,user_id,skill_id,correct\n"
        for text in ["order_id,user_id\n1,2\n", header + "1,2,3,yes\n", header + "1,2,3,1\n1,4,3,0\n", header]:
            with self.subTest(text=text), self.assertRaises(ValueError):
                load_assistments(io.StringIO(text))

    def test_folds_keep_learners_apart(self):
        folds = learner_folds(self.data["learners"], 5)
        self.assertEqual(len(folds), len(self.data["learners"]))
        self.assertTrue(set(folds.tolist()) <= set(range(5)))
        np.testing.assert_array_equal(folds, learner_folds(self.data["learners"], 5))
        with self.assertRaises(ValueError):
            learner_folds(self.data["learners"], 1)

    def _python_sequences(self, mask):
        d, out = self.data, {}
        for i in np.flatnonzero(mask):
            out.setdefault(int(d["skill"][i]), {}).setdefault(int(d["learner"][i]), []).append(bool(d["correct"][i]))
        return out

    def test_vectorised_em_matches_the_reviewed_python_fit(self):
        mask = np.ones(len(self.data["correct"]), dtype=bool)
        start = BATCH_STARTS[0]
        batch = fit_bkt_batch(self.data, mask, iterations=5, tolerance=-np.inf, starts=[start])
        for skill, by_learner in self._python_sequences(mask).items():
            reference, _ = _em(list(by_learner.values()), dict(start), 5, -math.inf)
            for name, value in reference.items():
                with self.subTest(skill=skill, param=name):
                    self.assertAlmostEqual(batch[name][skill], value, places=9)

    def test_vectorised_prediction_matches_the_reviewed_python_prediction(self):
        mask = np.ones(len(self.data["correct"]), dtype=bool)
        params = fit_bkt_batch(self.data, mask, iterations=10)
        predicted = predict_bkt_batch(params, self.data, mask)
        d = self.data
        for skill in range(len(d["skills"])):
            for learner in set(d["learner"][d["skill"] == skill].tolist()):
                idx = np.flatnonzero((d["skill"] == skill) & (d["learner"] == learner))
                reference, _ = predict_sequence({n: float(params[n][skill]) for n in BKT_BOUNDS},
                                                [bool(v) for v in d["correct"][idx]])
                np.testing.assert_allclose(predicted[idx], reference, rtol=1e-12)

    def test_forgetting_stays_in_bounds_and_plain_bkt_never_forgets(self):
        mask = np.ones(len(self.data["correct"]), dtype=bool)
        plain = fit_bkt_batch(self.data, mask, iterations=10)
        forgetting = fit_bkt_batch(self.data, mask, forgetting=True, iterations=10)
        self.assertTrue(np.all(plain["p_forget"] == 0))
        self.assertTrue(np.all((forgetting["p_forget"] >= 0) & (forgetting["p_forget"] <= FORGET_BOUNDS[1])))

    def test_pfa_counts_only_earlier_answers_on_the_same_skill(self):
        successes, failures = pfa_features(self.data)
        first = np.r_[True, np.diff(self.data["learner"]) != 0]
        self.assertTrue(np.all(successes[first] == 0) and np.all(failures[first] == 0))
        d = self.data
        i = int(np.flatnonzero(d["position"] == 5)[0])
        same = np.flatnonzero((d["learner"] == d["learner"][i]) & (d["skill"] == d["skill"][i])
                              & (d["position"] < d["position"][i]))
        self.assertEqual(successes[i], d["correct"][same].sum())
        self.assertEqual(failures[i], len(same) - d["correct"][same].sum())

    def test_auc_handles_perfect_reversed_and_tied_scores(self):
        self.assertEqual(auc([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]), 1.0)
        self.assertEqual(auc([0, 0, 1, 1], [0.9, 0.8, 0.2, 0.1]), 0.0)
        self.assertEqual(auc([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]), 0.5)
        self.assertTrue(math.isnan(auc([1, 1], [0.2, 0.3])))
        with self.assertRaises(ValueError):
            score(np.array([1.0]), np.array([np.nan]))

    def test_classic_benchmark_runs_and_bkt_beats_overall_rate(self):
        data = load_assistments(io.StringIO(fake_assistments_csv(n_learners=300, seed=5)))
        results, fits = run_classic_benchmark(data, folds=3, log=lambda _: None)
        summary = summarise(results)
        self.assertEqual(set(summary), {"Overall correct rate", "Per-skill correct rate",
                                        "BKT", "BKT + forgetting", "PFA"})
        self.assertGreater(summary["BKT"]["auc"]["mean"], summary["Overall correct rate"]["auc"]["mean"])
        ranges = parameter_ranges(fits, min_answers=50)
        self.assertLessEqual(ranges["p_slip"]["p10"], ranges["p_slip"]["p90"])

    @unittest.skipUnless(importlib.util.find_spec("torch"), "PyTorch not installed")
    def test_dkt_smoke_run(self):
        results = run_dkt(self.data, folds=2, hidden=8, epochs=2, patience=1, log=lambda _: None)
        self.assertEqual(len(results), 2)
        self.assertTrue(all(0 <= r["auc"] <= 1 for r in results))


# Runs in the notebook; test_bkt_benchmark.py loads this class without running it here.
if __name__ == "__main__":
    benchmark_result = unittest.TextTestRunner(verbosity=1).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(AssistmentsBenchmarkTests))
    if not benchmark_result.wasSuccessful():
        raise AssertionError("benchmark tests failed")
    print("Benchmark checks passed.")

In [ ]:
csv_path = download_assistments()
with open(csv_path, encoding="latin-1") as handle:
    real = load_assistments(handle)
print(ASSISTMENTS_CITATION)
print(f"REAL DATA: {len(real['correct']):,} answers from {len(real['learners']):,} learners on "
      f"{len(real['skills'])} skills; {real['dropped_without_skill']:,} rows without a skill dropped.")
print(f"Overall correct rate {real['correct'].mean():.3f}")

In [ ]:
classic_results, classic_bkt_fits = run_classic_benchmark(real, folds=5)

In [ ]:
import importlib.util

if importlib.util.find_spec("torch") is None:
    dkt_results = None
    print("PyTorch is not installed; skipping DKT.")
else:
    import torch
    if not torch.cuda.is_available():
        print("No GPU: DKT will be slow. Runtime → Change runtime type → T4 GPU, then Run all.")
    dkt_results = run_dkt(real, folds=5)

In [ ]:
all_results = dict(classic_results)
if dkt_results:
    all_results["DKT (LSTM)"] = dkt_results
benchmark_summary = summarise(all_results)
print(f"{'model':<24} {'AUC':>15} {'RMSE':>15} {'accuracy':>15}")
for name, metrics in sorted(benchmark_summary.items(), key=lambda item: item[1]["auc"]["mean"]):
    print(f"{name:<24} " + " ".join(f"{metrics[m]['mean']:.4f} ± {metrics[m]['std']:.4f}"
                                    for m in ("auc", "rmse", "accuracy")))

benchmark = {
    "schemaVersion": 1,
    "dataset": {"name": "ASSISTments 2009-2010 skill builder (corrected)", "citation": ASSISTMENTS_CITATION,
                "answers": int(len(real["correct"])), "learners": len(real["learners"]),
                "skills": len(real["skills"]), "droppedWithoutSkill": int(real["dropped_without_skill"]),
                "note": "Real US middle-school maths answers. Aggregate results only; the data itself is never committed."},
    "protocol": "5-fold cross-validation split by learner (SHA-256 of the id); each held-out learner's "
                "answers after their first are predicted one step ahead; DKT picks its epoch on 10% of "
                "training learners.",
    "models": benchmark_summary,
    "bktParameterRanges": parameter_ranges(classic_bkt_fits),
    "useInGullak": "Model choice and realistic BKT parameter ranges only. Maths skills are not money "
                   "concepts; Gullak keeps onboarding priors and quiz pass marks until its own data exists.",
}
benchmark_path = "knowledge-tracing-benchmark.assistments-2009.json"
with open(benchmark_path, "w", encoding="utf-8") as handle:
    json.dump(benchmark, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
print(f"\nWrote {benchmark_path}")
print(json.dumps(benchmark["bktParameterRanges"], indent=2))
try:
    from google.colab import files
    files.download(benchmark_path)
except ImportError:
    pass

## Next increment — not implemented

Later: fit on real Gullak quiz answers exported by R3 (outside this shared notebook), starting from the ASSISTments parameter ranges; review the parameters; then port `predict_sequence` to `packages/engine` and run it on device. BKT stays the on-device model (small, explainable, offline); DKT is the path once there is enough data. Initial app mastery priors are none=0.1, some=0.3, regular=0.5; weak <0.6, practising >=0.6 and <0.95, mastered >=0.95. These rules are recorded only, not applied in this notebook. Fitted parameters must not automatically replace app priors or influence production recommendations.

Do not describe synthetic results as real-student accuracy, or ASSISTments results as accuracy on Gullak's money topics.